# Derm7pt DenseNet121 with Cross-Validation and Explainable AI

**Dataset:** Derm7pt  
**Task:** Skin lesion classification  
**Model:** DenseNet121  
**Methods:** Five-fold stratified cross-validation, Grad-CAM++, GradientSHAP and DeepLiftSHAP

This notebook evaluates DenseNet121 across stratified folds and analyses model predictions using complementary explainability techniques. It was developed in Google Colab; dataset, checkpoint and output paths must be adjusted before execution.

In [ ]:


!pip install -q timm grad-cam captum openpyxl



import os
import glob
import copy
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

from captum.attr import GradientShap



if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive already mounted.")
else:
    from google.colab import drive
    drive.mount("/content/drive")



SEED = 42

DERM7PT_ROOT = "/content/drive/MyDrive/Derm7pt/release_v0"
META_PATH = "/content/drive/MyDrive/Derm7pt/release_v0/meta/meta.csv"

MODEL_NAME = "densenet121"
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0

EPOCHS = 12
N_SPLITS = 5

LR_HEAD = 3e-4
LR_FULL = 8e-5
WEIGHT_DECAY = 1e-4
USE_AMP = True
WARMUP_HEAD_EPOCHS = 2


USE_BALANCED_SUBSET = True
NEGATIVE_MULTIPLIER = 4

OUTPUT_DIR = "/content/drive/MyDrive/DERM7PT_FULL_DENSENET121_5FOLD_XAI"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOCAL_ORIGINAL_DIR = "/content/derm7pt_full_original"
LOCAL_CROP_DIR = "/content/derm7pt_full_crop"

os.makedirs(LOCAL_ORIGINAL_DIR, exist_ok=True)
os.makedirs(LOCAL_CROP_DIR, exist_ok=True)

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
XAI_DIR = os.path.join(OUTPUT_DIR, "xai_outputs")
CROP_DEBUG_DIR = os.path.join(OUTPUT_DIR, "crop_examples")

GRADCAMPP_DIR = os.path.join(XAI_DIR, "gradcampp_best_fold")
ACROSS_EPOCHS_DIR = os.path.join(XAI_DIR, "gradcampp_across_epochs_best_fold")
SHAP_DIR = os.path.join(XAI_DIR, "gradient_shap_best_fold")

for d in [
    CHECKPOINT_DIR,
    XAI_DIR,
    CROP_DEBUG_DIR,
    GRADCAMPP_DIR,
    ACROSS_EPOCHS_DIR,
    SHAP_DIR
]:
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Output:", OUTPUT_DIR)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)



print("DERM7PT_ROOT exists:", os.path.exists(DERM7PT_ROOT))
print("META_PATH exists:", os.path.exists(META_PATH))

if not os.path.exists(META_PATH):
    raise FileNotFoundError(f"Δεν βρέθηκε το meta.csv: {META_PATH}")

df_meta = pd.read_csv(META_PATH)

print("\nOfficial metadata shape:", df_meta.shape)
print("Columns:")
print(df_meta.columns.tolist())
display(df_meta.head())



image_files = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    image_files.extend(
        glob.glob(os.path.join(DERM7PT_ROOT, "**", ext), recursive=True)
    )

print("\nTotal image files found under release_v0:", len(image_files))

image_index = {}

for p in image_files:
    base = os.path.basename(p)
    stem = os.path.splitext(base)[0]
    rel = os.path.relpath(p, DERM7PT_ROOT).replace("\\", "/")

    image_index[base.lower()] = p
    image_index[stem.lower()] = p
    image_index[rel.lower()] = p

def match_derm_path(x):
    if pd.isna(x):
        return None

    x = str(x).strip().replace("\\", "/")
    base = os.path.basename(x)
    stem = os.path.splitext(base)[0]

    candidates = [
        x.lower(),
        base.lower(),
        stem.lower(),
        ("images/" + x).lower(),
        ("release_v0/images/" + x).lower()
    ]

    for c in candidates:
        if c in image_index:
            return image_index[c]

    possible_paths = [
        os.path.join(DERM7PT_ROOT, x),
        os.path.join(DERM7PT_ROOT, "images", x),
        os.path.join(DERM7PT_ROOT, "release_v0", "images", x)
    ]

    for p in possible_paths:
        if os.path.exists(p):
            return p

    return None

df = df_meta.copy()
df["image_path"] = df["derm"].apply(match_derm_path)

print("\nAfter derm image matching:")
print("Rows:", len(df))
print("Matched images:", df["image_path"].notna().sum())
print("Missing images:", df["image_path"].isna().sum())

if df["image_path"].isna().sum() > 0:
    print("\nMissing examples:")
    display(df[df["image_path"].isna()][["case_num", "diagnosis", "derm"]].head(10))

df = df.dropna(subset=["image_path"]).reset_index(drop=True)



def make_binary_label_from_diagnosis(x):
    s = str(x).lower().strip()
    if "melanoma" in s:
        return 1
    return 0

df["binary_label"] = df["diagnosis"].apply(make_binary_label_from_diagnosis).astype(int)
df["binary_class"] = df["binary_label"].map({
    0: "non_melanoma",
    1: "melanoma"
})

print("\nDiagnosis distribution:")
print(df["diagnosis"].value_counts())

print("\nBinary distribution before optional balancing:")
print(df["binary_class"].value_counts())



if USE_BALANCED_SUBSET:
    df_pos = df[df["binary_label"] == 1].copy()
    df_neg = df[df["binary_label"] == 0].copy()

    n_pos = len(df_pos)
    n_neg = min(len(df_neg), n_pos * NEGATIVE_MULTIPLIER)

    df_neg = df_neg.sample(n=n_neg, random_state=SEED)

    df = pd.concat([df_pos, df_neg], axis=0)
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    print("\nUsing balanced/subsampled dataset:")
    print("Melanoma:", n_pos)
    print("Non-melanoma sampled:", n_neg)

else:
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print("\nUsing full matched dataset without negative subsampling.")

print("\nFinal binary distribution:")
print(df["binary_class"].value_counts())
print("Final df shape:", df.shape)



def read_rgb_cv2(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)

    if img is None:
        raise ValueError(f"cv2.imread returned None: {path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def remove_black_borders(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    mask = gray > 12

    if mask.sum() < 100:
        return rgb

    ys, xs = np.where(mask)

    y1, y2 = ys.min(), ys.max()
    x1, x2 = xs.min(), xs.max()

    return rgb[y1:y2 + 1, x1:x2 + 1]

def lesion_focused_crop(rgb, pad_ratio=0.30, min_area_ratio=0.004):
    rgb_clean = remove_black_borders(rgb)

    h, w = rgb_clean.shape[:2]

    if h < 50 or w < 50:
        return rgb_clean

    hsv = cv2.cvtColor(rgb_clean, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1].astype(np.float32)
    V = hsv[:, :, 2].astype(np.float32)

    darkness = 255.0 - V
    saturation = S

    score = 0.70 * darkness + 0.30 * saturation
    score = cv2.GaussianBlur(score, (9, 9), 0)
    score_uint = np.clip(score, 0, 255).astype(np.uint8)

    _, mask = cv2.threshold(
        score_uint,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = np.ones((9, 9), np.uint8)

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return rgb_clean

    areas = [cv2.contourArea(c) for c in contours]
    max_idx = int(np.argmax(areas))
    max_area = areas[max_idx]

    if max_area < min_area_ratio * h * w:
        return rgb_clean

    x, y, bw, bh = cv2.boundingRect(contours[max_idx])

    pad_x = int(bw * pad_ratio)
    pad_y = int(bh * pad_ratio)

    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(w, x + bw + pad_x)
    y2 = min(h, y + bh + pad_y)

    crop = rgb_clean[y1:y2, x1:x2]

    if crop.shape[0] < 40 or crop.shape[1] < 40:
        return rgb_clean

    return crop

def save_rgb(path, rgb):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, bgr)



print("\nCreating local original/crop dataset...")

original_paths = []
crop_paths = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    src = row["image_path"]
    cls = row["binary_class"]
    fname = os.path.basename(src)

    orig_dir = os.path.join(LOCAL_ORIGINAL_DIR, cls)
    crop_dir = os.path.join(LOCAL_CROP_DIR, cls)

    os.makedirs(orig_dir, exist_ok=True)
    os.makedirs(crop_dir, exist_ok=True)

    orig_dst = os.path.join(orig_dir, fname)
    crop_dst = os.path.join(crop_dir, fname)

    if not os.path.exists(orig_dst):
        shutil.copy2(src, orig_dst)

    if not os.path.exists(crop_dst):
        try:
            rgb = read_rgb_cv2(src)
            crop = lesion_focused_crop(rgb)
            save_rgb(crop_dst, crop)
        except Exception as e:
            print("Crop error:", src, e)
            shutil.copy2(src, crop_dst)

    original_paths.append(orig_dst)
    crop_paths.append(crop_dst)

df["original_local_path"] = original_paths
df["image_path"] = crop_paths

fixed_df_path = os.path.join(OUTPUT_DIR, "derm7pt_full_binary_fixed_used.csv")
df.to_csv(fixed_df_path, index=False)

print("\nSaved fixed dataframe:", fixed_df_path)


debug_df = df.sample(n=min(10, len(df)), random_state=SEED).reset_index(drop=True)

plt.figure(figsize=(12, 3 * len(debug_df)))

for i, row in debug_df.iterrows():
    orig = Image.open(row["original_local_path"]).convert("RGB")
    crop = Image.open(row["image_path"]).convert("RGB")

    plt.subplot(len(debug_df), 2, 2 * i + 1)
    plt.imshow(orig)
    plt.axis("off")
    plt.title(f"Original | {row['binary_class']}")

    plt.subplot(len(debug_df), 2, 2 * i + 2)
    plt.imshow(crop)
    plt.axis("off")
    plt.title("Lesion-focused crop")

plt.tight_layout()

crop_examples_path = os.path.join(CROP_DEBUG_DIR, "crop_examples.png")
plt.savefig(crop_examples_path, dpi=250, bbox_inches="tight")
plt.show()

print("Saved crop examples:", crop_examples_path)



def safe_open_rgb(path):
    try:
        img = cv2.imread(path, cv2.IMREAD_COLOR)

        if img is None:
            raise ValueError("cv2.imread returned None")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    except Exception as e:
        print("Image error:", path, e)
        arr = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        return Image.fromarray(arr)

train_tf = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.90, 1.10)
    ),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=12),
    T.ColorJitter(
        brightness=0.08,
        contrast=0.10,
        saturation=0.08,
        hue=0.015
    ),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

class Derm7ptBinaryDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = safe_open_rgb(row["image_path"])

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(float(row["binary_label"]), dtype=torch.float32)

        return image, label



def create_model(pretrained=True):
    model = timm.create_model(
        MODEL_NAME,
        pretrained=pretrained,
        num_classes=1
    )
    return model

class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, alpha=0.70, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
            pos_weight=self.pos_weight
        )

        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)

        alpha_t = torch.where(
            targets == 1,
            torch.tensor(self.alpha, device=targets.device),
            torch.tensor(1 - self.alpha, device=targets.device)
        )

        loss = alpha_t * ((1 - pt) ** self.gamma) * bce

        return loss.mean()

def set_head_only(model):
    for p in model.parameters():
        p.requires_grad = False

    if hasattr(model, "classifier"):
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif hasattr(model, "head"):
        for p in model.head.parameters():
            p.requires_grad = True
    else:
        for name, p in model.named_parameters():
            if "classifier" in name or "head" in name:
                p.requires_grad = True

def set_full_trainable(model):
    for p in model.parameters():
        p.requires_grad = True

def evaluate_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "pred": y_pred
    }

def find_best_threshold(y_true, y_prob, metric="f1"):
    thresholds = np.linspace(0.10, 0.90, 81)

    best_threshold = 0.5
    best_score = -1

    for th in thresholds:
        m = evaluate_at_threshold(y_true, y_prob, th)
        score = m[metric]

        if score > best_score:
            best_score = score
            best_threshold = th

    return best_threshold, best_score

@torch.no_grad()
def get_probs_labels(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    for images, labels in tqdm(loader, desc="Predicting", leave=False):
        images = images.to(device)

        logits = model(images).view(-1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        probs_all.extend(probs)
        labels_all.extend(labels.numpy())

    return np.array(labels_all).astype(int), np.array(probs_all)



def train_one_fold(fold, train_fold_df, val_fold_df):
    print("\n" + "=" * 90)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 90)

    print("Train distribution:")
    print(train_fold_df["binary_class"].value_counts())

    print("\nValidation distribution:")
    print(val_fold_df["binary_class"].value_counts())

    fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    fold_ckpt_dir = os.path.join(CHECKPOINT_DIR, f"fold_{fold}")

    os.makedirs(fold_dir, exist_ok=True)
    os.makedirs(fold_ckpt_dir, exist_ok=True)

    train_ds = Derm7ptBinaryDataset(train_fold_df, transform=train_tf)
    val_ds = Derm7ptBinaryDataset(val_fold_df, transform=eval_tf)

    train_labels = train_fold_df["binary_label"].values.astype(int)

    class_counts = np.bincount(train_labels, minlength=2)
    class_counts = np.maximum(class_counts, 1)

    sample_weights = (1.0 / class_counts)[train_labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

    model = create_model(pretrained=True).to(device)

    pos = int(train_fold_df["binary_label"].sum())
    neg = int((train_fold_df["binary_label"] == 0).sum())

    pos_weight = torch.tensor(
        [neg / max(pos, 1)],
        dtype=torch.float32
    ).to(device)

    criterion = FocalBCEWithLogitsLoss(
        alpha=0.70,
        gamma=2.0,
        pos_weight=pos_weight
    )

    set_head_only(model)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=(USE_AMP and device.type == "cuda")
    )

    best_val_auc = -1
    best_state = None
    best_epoch = -1
    best_threshold = 0.5

    history = []

    for epoch in range(1, EPOCHS + 1):

        if epoch == WARMUP_HEAD_EPOCHS + 1:
            print("\nUnfreezing full DenseNet121 model...")

            set_full_trainable(model)

            optimizer = optim.AdamW(
                model.parameters(),
                lr=LR_FULL,
                weight_decay=WEIGHT_DECAY
            )

            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=EPOCHS - WARMUP_HEAD_EPOCHS
            )

        model.train()

        running_loss = 0.0
        n = 0

        loop = tqdm(
            train_loader,
            desc=f"Fold {fold} | DenseNet121 Epoch {epoch}/{EPOCHS}"
        )

        for images, labels in loop:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits = model(images).view(-1)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            n += images.size(0)

            loop.set_postfix(loss=f"{loss.item():.4f}")

        scheduler.step()

        train_loss = running_loss / max(n, 1)

        y_val, p_val = get_probs_labels(model, val_loader)
        threshold, _ = find_best_threshold(y_val, p_val, metric="f1")
        val_metrics = evaluate_at_threshold(y_val, p_val, threshold)

        print(
            f"Fold {fold} | Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_precision={val_metrics['precision']:.4f} | "
            f"val_recall={val_metrics['recall']:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['auc']:.4f} | "
            f"thr={threshold:.2f}"
        )

        history.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_auc": val_metrics["auc"],
            "threshold": threshold
        })

        epoch_ckpt_path = os.path.join(
            fold_ckpt_dir,
            f"fold_{fold}_epoch_{epoch}.pth"
        )

        torch.save({
            "fold": fold,
            "epoch": epoch,
            "model_state": copy.deepcopy(model.state_dict()),
            "threshold": threshold,
            "val_auc": val_metrics["auc"],
            "val_f1": val_metrics["f1"],
            "model_name": MODEL_NAME,
            "img_size": IMG_SIZE
        }, epoch_ckpt_path)

        if not np.isnan(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc:
            best_val_auc = val_metrics["auc"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            best_threshold = threshold

            best_ckpt_path = os.path.join(
                fold_ckpt_dir,
                f"best_fold_{fold}.pth"
            )

            torch.save({
                "fold": fold,
                "epoch": best_epoch,
                "model_state": best_state,
                "threshold": best_threshold,
                "best_val_auc": best_val_auc,
                "model_name": MODEL_NAME,
                "img_size": IMG_SIZE
            }, best_ckpt_path)

            print("Saved best model:", best_ckpt_path)

    history_df = pd.DataFrame(history)

    history_path = os.path.join(fold_dir, f"history_fold_{fold}.csv")
    history_df.to_csv(history_path, index=False)

    if best_state is None:
        raise RuntimeError(f"No best state for fold {fold}")

    model.load_state_dict(best_state)
    model.eval()

    y_val, p_val = get_probs_labels(model, val_loader)
    final_metrics = evaluate_at_threshold(y_val, p_val, best_threshold)

    print("\n=== FINAL FOLD RESULTS ===")
    print("Fold:", fold)
    print("Best epoch:", best_epoch)
    print("Threshold:", best_threshold)
    print("Accuracy:", final_metrics["accuracy"])
    print("Precision:", final_metrics["precision"])
    print("Recall:", final_metrics["recall"])
    print("F1:", final_metrics["f1"])
    print("AUC:", final_metrics["auc"])

    print("\nClassification Report:")
    print(
        classification_report(
            y_val,
            final_metrics["pred"],
            target_names=["non_melanoma", "melanoma"],
            zero_division=0
        )
    )

    cm = confusion_matrix(y_val, final_metrics["pred"])

    print("\nConfusion Matrix:")
    print(cm)

    fold_predictions_df = val_fold_df.copy().reset_index(drop=True)
    fold_predictions_df["true_label"] = y_val
    fold_predictions_df["prob_melanoma"] = p_val
    fold_predictions_df["pred_label"] = final_metrics["pred"]
    fold_predictions_df["true_class"] = fold_predictions_df["true_label"].map({
        0: "non_melanoma",
        1: "melanoma"
    })
    fold_predictions_df["pred_class"] = fold_predictions_df["pred_label"].map({
        0: "non_melanoma",
        1: "melanoma"
    })
    fold_predictions_df["correct"] = fold_predictions_df["true_label"] == fold_predictions_df["pred_label"]

    predictions_path = os.path.join(fold_dir, f"predictions_fold_{fold}.csv")
    fold_predictions_df.to_csv(predictions_path, index=False)

    plt.figure(figsize=(5.5, 4.8))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix - Fold {fold}")
    plt.colorbar()

    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["non_melanoma", "melanoma"])
    plt.yticks(tick_marks, ["non_melanoma", "melanoma"])

    plt.xlabel("Predicted")
    plt.ylabel("True")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()

    cm_path = os.path.join(fold_dir, f"confusion_matrix_fold_{fold}.png")
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()

    result_row = {
        "fold": fold,
        "best_epoch": best_epoch,
        "threshold": best_threshold,
        "accuracy": final_metrics["accuracy"],
        "precision": final_metrics["precision"],
        "recall": final_metrics["recall"],
        "f1": final_metrics["f1"],
        "auc": final_metrics["auc"],
        "history_path": history_path,
        "predictions_path": predictions_path,
        "confusion_matrix_path": cm_path,
        "best_checkpoint_path": os.path.join(fold_ckpt_dir, f"best_fold_{fold}.pth"),
        "fold_ckpt_dir": fold_ckpt_dir
    }

    del model
    torch.cuda.empty_cache()

    return result_row


skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

all_fold_results = []

X = df.index.values
y = df["binary_label"].values.astype(int)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    train_fold_df = df.iloc[train_idx].reset_index(drop=True)
    val_fold_df = df.iloc[val_idx].reset_index(drop=True)

    result = train_one_fold(
        fold=fold,
        train_fold_df=train_fold_df,
        val_fold_df=val_fold_df
    )

    all_fold_results.append(result)

    pd.DataFrame(all_fold_results).to_csv(
        os.path.join(OUTPUT_DIR, "intermediate_cv_results.csv"),
        index=False
    )



cv_results_df = pd.DataFrame(all_fold_results)

cv_results_path = os.path.join(OUTPUT_DIR, "derm7pt_full_5fold_cv_results.csv")
cv_results_df.to_csv(cv_results_path, index=False)

print("\n=== 5-FOLD CV RESULTS ===")
display(cv_results_df)

metrics_cols = ["accuracy", "precision", "recall", "f1", "auc"]

summary_rows = []

for m in metrics_cols:
    summary_rows.append({
        "metric": m,
        "mean": cv_results_df[m].mean(),
        "std": cv_results_df[m].std(),
        "min": cv_results_df[m].min(),
        "max": cv_results_df[m].max()
    })

cv_summary_df = pd.DataFrame(summary_rows)

cv_summary_path = os.path.join(OUTPUT_DIR, "derm7pt_full_5fold_cv_summary.csv")
cv_summary_df.to_csv(cv_summary_path, index=False)

print("\n=== 5-FOLD CV SUMMARY ===")
display(cv_summary_df)

plt.figure(figsize=(8, 5))

means = cv_summary_df["mean"].values * 100
stds = cv_summary_df["std"].values * 100
labels = cv_summary_df["metric"].values

plt.bar(labels, means, yerr=stds, capsize=5)
plt.ylabel("Score (%)")
plt.title("Derm7pt DenseNet121 - 5-Fold Cross-Validation")
plt.grid(axis="y")

cv_barplot_path = os.path.join(OUTPUT_DIR, "derm7pt_full_5fold_cv_barplot.png")
plt.savefig(cv_barplot_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved CV results:", cv_results_path)
print("Saved CV summary:", cv_summary_path)
print("Saved barplot:", cv_barplot_path)



label_name_map = {
    0: "non_melanoma",
    1: "melanoma"
}

best_fold_row = cv_results_df.sort_values("auc", ascending=False).iloc[0]

BEST_FOLD = int(best_fold_row["fold"])
BEST_FOLD_CKPT = best_fold_row["best_checkpoint_path"]
BEST_FOLD_PRED_PATH = best_fold_row["predictions_path"]
BEST_FOLD_CKPT_DIR = best_fold_row["fold_ckpt_dir"]

print("\nBest fold for XAI:", BEST_FOLD)
print("Best fold checkpoint:", BEST_FOLD_CKPT)
print("Best fold predictions:", BEST_FOLD_PRED_PATH)

test_results_df = pd.read_csv(BEST_FOLD_PRED_PATH)

def get_target_layer_densenet(model):
    if hasattr(model, "features"):
        return model.features.denseblock4

    if hasattr(model, "blocks"):
        return model.blocks[-1]

    raise AttributeError("Δεν βρέθηκε target layer για DenseNet.")

def load_rgb_np_and_tensor(img_path):
    pil_img = safe_open_rgb(img_path)
    pil_show = pil_img.resize((IMG_SIZE, IMG_SIZE))

    rgb_np = np.array(pil_show).astype(np.float32) / 255.0
    x = eval_tf(pil_img).unsqueeze(0).to(device)

    return pil_show, rgb_np, x

class BinaryCAMWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

def load_model_from_checkpoint(ckpt_path):
    # Safe only for checkpoints created by this notebook; never load untrusted files.
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    model = create_model(pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    return model, ckpt

best_model, best_ckpt = load_model_from_checkpoint(BEST_FOLD_CKPT)

def select_xai_cases(pred_df):
    dfp = pred_df.copy()

    dfp["pred_confidence"] = np.where(
        dfp["pred_label"] == 1,
        dfp["prob_melanoma"],
        1.0 - dfp["prob_melanoma"]
    )

    correct_melanoma = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    correct_non_melanoma = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    wrong = dfp[dfp["correct"] == False].copy()

    cases = {}

    if len(correct_melanoma) > 0:
        cases["correct_melanoma"] = correct_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(correct_non_melanoma) > 0:
        cases["correct_non_melanoma"] = correct_non_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(wrong) > 0:
        cases["wrong_prediction"] = wrong.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    return cases

xai_cases = select_xai_cases(test_results_df)

print("\nSelected XAI cases:")
print(xai_cases)



def plot_gradcampp(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]
    original_path = row["original_local_path"] if "original_local_path" in row else row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])
    correct = bool(row["correct"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    pil_crop, rgb_np, x = load_rgb_np_and_tensor(img_path)
    pil_original = safe_open_rgb(original_path)

    cam_model = BinaryCAMWrapper(best_model).to(device).eval()
    target_layer = get_target_layer_densenet(best_model)

    cam = GradCAMPlusPlus(
        model=cam_model,
        target_layers=[target_layer]
    )

    grayscale_cam = cam(
        input_tensor=x,
        targets=[ClassifierOutputTarget(pred_label)]
    )[0]

    overlay = show_cam_on_image(
        rgb_np,
        grayscale_cam,
        use_rgb=True,
        image_weight=0.78
    )

    plt.figure(figsize=(18, 4.8))

    plt.subplot(1, 4, 1)
    plt.imshow(pil_original)
    plt.axis("off")
    plt.title(f"Original\nTrue={true_class}")

    plt.subplot(1, 4, 2)
    plt.imshow(pil_crop)
    plt.axis("off")
    plt.title("Lesion crop")

    plt.subplot(1, 4, 3)
    plt.imshow(grayscale_cam, cmap="jet")
    plt.axis("off")
    plt.title("Grad-CAM++")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(
        f"Overlay on crop\n"
        f"Pred={pred_class} | Correct={correct}\n"
        f"Prob melanoma={prob:.3f}"
    )

    plt.tight_layout()

    save_path = os.path.join(
        GRADCAMPP_DIR,
        f"fold_{BEST_FOLD}_{case_name}_gradcampp.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_gradcampp(idx, case_name)



EPOCHS_TO_SHOW = [1, 5, 10]

def compute_cam_from_checkpoint(ckpt_path, img_path, target_class):
    temp_model, _ = load_model_from_checkpoint(ckpt_path)

    cam_model = BinaryCAMWrapper(temp_model).to(device).eval()

    pil_img, rgb_np, x = load_rgb_np_and_tensor(img_path)

    target_layer = get_target_layer_densenet(temp_model)

    cam = GradCAMPlusPlus(
        model=cam_model,
        target_layers=[target_layer]
    )

    grayscale_cam = cam(
        input_tensor=x,
        targets=[ClassifierOutputTarget(target_class)]
    )[0]

    overlay = show_cam_on_image(
        rgb_np,
        grayscale_cam,
        use_rgb=True,
        image_weight=0.78
    )

    del temp_model
    torch.cuda.empty_cache()

    return overlay

def plot_across_epochs(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    target_class = true_label

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))

    plot_items = []

    for ep in EPOCHS_TO_SHOW:
        ckpt_path = os.path.join(
            BEST_FOLD_CKPT_DIR,
            f"fold_{BEST_FOLD}_epoch_{ep}.pth"
        )

        if os.path.exists(ckpt_path):
            plot_items.append((f"Epoch {ep}", ckpt_path))
        else:
            print("Missing checkpoint:", ckpt_path)

    plot_items.append((f"Best epoch {best_ckpt.get('epoch', 'best')}", BEST_FOLD_CKPT))

    n_cols = len(plot_items) + 1

    plt.figure(figsize=(4.3 * n_cols, 4.8))

    plt.subplot(1, n_cols, 1)
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title(
        f"Crop input\n"
        f"True={true_class}\n"
        f"Pred={pred_class}\n"
        f"Prob={prob:.3f}"
    )

    for col_idx, (title, ckpt_path) in enumerate(plot_items, start=2):
        overlay = compute_cam_from_checkpoint(
            ckpt_path=ckpt_path,
            img_path=img_path,
            target_class=target_class
        )

        plt.subplot(1, n_cols, col_idx)
        plt.imshow(overlay)
        plt.axis("off")
        plt.title(title)

    plt.suptitle(
        f"Grad-CAM++ Across Epochs - Fold {BEST_FOLD} - {case_name}\n"
        f"Target class: {true_class}",
        fontsize=14
    )

    plt.tight_layout()

    save_path = os.path.join(
        ACROSS_EPOCHS_DIR,
        f"fold_{BEST_FOLD}_{case_name}_gradcampp_across_epochs.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_across_epochs(idx, case_name)



class BinaryTwoOutputWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

def compute_gradient_shap(input_tensor, target_class):
    wrapped_model = BinaryTwoOutputWrapper(best_model).to(device).eval()
    gradient_shap = GradientShap(wrapped_model)

    baseline_black = torch.zeros_like(input_tensor).to(device)
    baseline_noise = torch.randn_like(input_tensor).to(device) * 0.05

    baselines = torch.cat([baseline_black, baseline_noise], dim=0)

    attr = gradient_shap.attribute(
        input_tensor,
        baselines=baselines,
        target=target_class,
        n_samples=30,
        stdevs=0.08
    )

    attr = attr.detach().cpu()[0]

    signed_map = attr.mean(dim=0).numpy()
    abs_map = attr.abs().mean(dim=0).numpy()

    signed_map = cv2.GaussianBlur(signed_map, (7, 7), 0)
    abs_map = cv2.GaussianBlur(abs_map, (7, 7), 0)

    abs_map = (abs_map - abs_map.min()) / (abs_map.max() - abs_map.min() + 1e-8)

    return signed_map, abs_map

def plot_gradient_shap(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    target_class = pred_label
    target_name = label_name_map[target_class]

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(pil_img).astype(np.float32) / 255.0

    input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)

    signed_map, abs_map = compute_gradient_shap(
        input_tensor=input_tensor,
        target_class=target_class
    )

    signed_absmax = np.max(np.abs(signed_map)) + 1e-12

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.axis("off")
    plt.title(f"Crop input\nTrue={true_class}\nPred={pred_class}")

    plt.subplot(1, 4, 2)
    plt.imshow(
        signed_map,
        cmap="bwr",
        vmin=-signed_absmax,
        vmax=signed_absmax
    )
    plt.axis("off")
    plt.title(f"Gradient SHAP signed\nExplaining={target_name}")

    plt.subplot(1, 4, 3)
    plt.imshow(abs_map, cmap="jet")
    plt.axis("off")
    plt.title("Absolute importance")

    plt.subplot(1, 4, 4)
    plt.imshow(img_np)
    plt.imshow(abs_map, cmap="jet", alpha=0.35)
    plt.axis("off")
    plt.title(f"Overlay\nProb melanoma={prob:.3f}")

    plt.tight_layout()

    save_path = os.path.join(
        SHAP_DIR,
        f"fold_{BEST_FOLD}_{case_name}_gradient_shap.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_gradient_shap(idx, case_name)



print("\nDerm7pt full DenseNet121 5-fold CV + XAI completed.")

print("\nDataset used:")
print("Fixed dataframe:", fixed_df_path)
print("Binary distribution:")
print(df["binary_class"].value_counts())

print("\nCV outputs:")
print("CV results:", cv_results_path)
print("CV summary:", cv_summary_path)
print("CV barplot:", cv_barplot_path)

print("\nBest fold for XAI:", BEST_FOLD)
print("Best fold checkpoint:", BEST_FOLD_CKPT)

print("\nXAI outputs:")
print("Grad-CAM++:", GRADCAMPP_DIR)
print("Grad-CAM++ across epochs:", ACROSS_EPOCHS_DIR)
print("Gradient SHAP:", SHAP_DIR)

print("\nMain output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# GRAD-CAM++ FOR BEST FOLD
# Derm7pt full + DenseNet121 + 5-fold CV
#
# Run after the 5-fold CV experiment.
# ============================================================

!pip install -q grad-cam

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

GRADCAMPP_DIR = os.path.join(OUTPUT_DIR, "xai_outputs", "gradcampp_best_fold_separate")
os.makedirs(GRADCAMPP_DIR, exist_ok=True)

print("Grad-CAM++ folder:", GRADCAMPP_DIR)

# ------------------------------------------------------------
# Label map
# ------------------------------------------------------------

label_name_map = {
    0: "non_melanoma",
    1: "melanoma"
}

# ------------------------------------------------------------
# Select best fold by AUC
# ------------------------------------------------------------

best_fold_row = cv_results_df.sort_values("auc", ascending=False).iloc[0]

BEST_FOLD = int(best_fold_row["fold"])
BEST_FOLD_CKPT = best_fold_row["best_checkpoint_path"]
BEST_FOLD_PRED_PATH = best_fold_row["predictions_path"]

print("Best fold:", BEST_FOLD)
print("Best checkpoint:", BEST_FOLD_CKPT)
print("Predictions:", BEST_FOLD_PRED_PATH)

test_results_df = pd.read_csv(BEST_FOLD_PRED_PATH)

# ------------------------------------------------------------
# Load best fold model
# ------------------------------------------------------------

def load_model_from_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    model = create_model(pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    return model, ckpt

best_model, best_ckpt = load_model_from_checkpoint(BEST_FOLD_CKPT)

print("Loaded best model from epoch:", best_ckpt.get("epoch", "unknown"))

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

class BinaryCAMWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

def get_target_layer_densenet(model):
    if hasattr(model, "features"):
        return model.features.denseblock4

    if hasattr(model, "blocks"):
        return model.blocks[-1]

    raise AttributeError("Δεν βρέθηκε target layer για DenseNet.")

def load_rgb_np_and_tensor(img_path):
    pil_img = safe_open_rgb(img_path)
    pil_show = pil_img.resize((IMG_SIZE, IMG_SIZE))

    rgb_np = np.array(pil_show).astype(np.float32) / 255.0
    x = eval_tf(pil_img).unsqueeze(0).to(device)

    return pil_show, rgb_np, x

def select_xai_cases(pred_df):
    dfp = pred_df.copy()

    dfp["pred_confidence"] = np.where(
        dfp["pred_label"] == 1,
        dfp["prob_melanoma"],
        1.0 - dfp["prob_melanoma"]
    )

    correct_melanoma = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    correct_non_melanoma = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    wrong = dfp[dfp["correct"] == False].copy()

    cases = {}

    if len(correct_melanoma) > 0:
        cases["correct_melanoma"] = correct_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(correct_non_melanoma) > 0:
        cases["correct_non_melanoma"] = correct_non_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(wrong) > 0:
        cases["wrong_prediction"] = wrong.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    return cases

xai_cases = select_xai_cases(test_results_df)

print("Selected Grad-CAM++ cases:")
print(xai_cases)

# ------------------------------------------------------------
# Grad-CAM++ plot
# ------------------------------------------------------------

def plot_gradcampp(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    if "original_local_path" in row and isinstance(row["original_local_path"], str):
        original_path = row["original_local_path"]
    else:
        original_path = img_path

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])
    correct = bool(row["correct"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    pil_crop, rgb_np, x = load_rgb_np_and_tensor(img_path)
    pil_original = safe_open_rgb(original_path)

    cam_model = BinaryCAMWrapper(best_model).to(device).eval()
    target_layer = get_target_layer_densenet(best_model)

    cam = GradCAMPlusPlus(
        model=cam_model,
        target_layers=[target_layer]
    )

    grayscale_cam = cam(
        input_tensor=x,
        targets=[ClassifierOutputTarget(pred_label)]
    )[0]

    overlay = show_cam_on_image(
        rgb_np,
        grayscale_cam,
        use_rgb=True,
        image_weight=0.78
    )

    plt.figure(figsize=(18, 4.8))

    plt.subplot(1, 4, 1)
    plt.imshow(pil_original)
    plt.axis("off")
    plt.title(f"Original\nTrue={true_class}")

    plt.subplot(1, 4, 2)
    plt.imshow(pil_crop)
    plt.axis("off")
    plt.title("Lesion crop / input")

    plt.subplot(1, 4, 3)
    plt.imshow(grayscale_cam, cmap="jet")
    plt.axis("off")
    plt.title("Grad-CAM++")

    plt.subplot(1, 4, 4)
    plt.imshow(overlay)
    plt.axis("off")
    plt.title(
        f"Overlay\n"
        f"Pred={pred_class} | Correct={correct}\n"
        f"Prob melanoma={prob:.3f}"
    )

    plt.tight_layout()

    save_path = os.path.join(
        GRADCAMPP_DIR,
        f"fold_{BEST_FOLD}_{case_name}_gradcampp.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_gradcampp(idx, case_name)

print("\nGrad-CAM++ completed.")


In [ ]:
# ============================================================
# GRAD-CAM++ ACROSS EPOCHS
# Epochs: 1, 5, 10, best
# Derm7pt full + DenseNet121 + 5-fold CV
# ============================================================

!pip install -q grad-cam

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

ACROSS_EPOCHS_DIR = os.path.join(OUTPUT_DIR, "xai_outputs", "gradcampp_across_epochs_separate")
os.makedirs(ACROSS_EPOCHS_DIR, exist_ok=True)

print("Across epochs folder:", ACROSS_EPOCHS_DIR)

# ------------------------------------------------------------
# Label map
# ------------------------------------------------------------

label_name_map = {
    0: "non_melanoma",
    1: "melanoma"
}

# ------------------------------------------------------------
# Select best fold
# ------------------------------------------------------------

best_fold_row = cv_results_df.sort_values("auc", ascending=False).iloc[0]

BEST_FOLD = int(best_fold_row["fold"])
BEST_FOLD_CKPT = best_fold_row["best_checkpoint_path"]
BEST_FOLD_PRED_PATH = best_fold_row["predictions_path"]
BEST_FOLD_CKPT_DIR = best_fold_row["fold_ckpt_dir"]

print("Best fold:", BEST_FOLD)
print("Best fold checkpoint:", BEST_FOLD_CKPT)
print("Best fold checkpoint dir:", BEST_FOLD_CKPT_DIR)
print("Predictions:", BEST_FOLD_PRED_PATH)

test_results_df = pd.read_csv(BEST_FOLD_PRED_PATH)

best_ckpt = torch.load(BEST_FOLD_CKPT, map_location=device, weights_only=False)
best_epoch = best_ckpt.get("epoch", "best")

print("Best epoch:", best_epoch)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

class BinaryCAMWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)
        return torch.stack([-logit, logit], dim=1)

def load_model_from_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    model = create_model(pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    return model, ckpt

def get_target_layer_densenet(model):
    if hasattr(model, "features"):
        return model.features.denseblock4

    if hasattr(model, "blocks"):
        return model.blocks[-1]

    raise AttributeError("Δεν βρέθηκε target layer για DenseNet.")

def load_rgb_np_and_tensor(img_path):
    pil_img = safe_open_rgb(img_path)
    pil_show = pil_img.resize((IMG_SIZE, IMG_SIZE))

    rgb_np = np.array(pil_show).astype(np.float32) / 255.0
    x = eval_tf(pil_img).unsqueeze(0).to(device)

    return pil_show, rgb_np, x

def select_xai_cases(pred_df):
    dfp = pred_df.copy()

    dfp["pred_confidence"] = np.where(
        dfp["pred_label"] == 1,
        dfp["prob_melanoma"],
        1.0 - dfp["prob_melanoma"]
    )

    correct_melanoma = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    correct_non_melanoma = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    wrong = dfp[dfp["correct"] == False].copy()

    cases = {}

    if len(correct_melanoma) > 0:
        cases["correct_melanoma"] = correct_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(correct_non_melanoma) > 0:
        cases["correct_non_melanoma"] = correct_non_melanoma.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    if len(wrong) > 0:
        cases["wrong_prediction"] = wrong.sort_values(
            "pred_confidence",
            ascending=False
        ).index[0]

    return cases

xai_cases = select_xai_cases(test_results_df)

print("Selected across-epochs cases:")
print(xai_cases)

# ------------------------------------------------------------
# Compute Grad-CAM++ from checkpoint
# ------------------------------------------------------------

def compute_cam_from_checkpoint(ckpt_path, img_path, target_class):
    temp_model, _ = load_model_from_checkpoint(ckpt_path)

    cam_model = BinaryCAMWrapper(temp_model).to(device).eval()

    pil_img, rgb_np, x = load_rgb_np_and_tensor(img_path)

    target_layer = get_target_layer_densenet(temp_model)

    cam = GradCAMPlusPlus(
        model=cam_model,
        target_layers=[target_layer]
    )

    grayscale_cam = cam(
        input_tensor=x,
        targets=[ClassifierOutputTarget(target_class)]
    )[0]

    overlay = show_cam_on_image(
        rgb_np,
        grayscale_cam,
        use_rgb=True,
        image_weight=0.78
    )

    del temp_model
    torch.cuda.empty_cache()

    return overlay

# ------------------------------------------------------------
# Across epochs plot
# ------------------------------------------------------------

EPOCHS_TO_SHOW = [1, 5, 10]

def plot_across_epochs(idx, case_name):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    # Για across epochs εξηγούμε την πραγματική κλάση, ώστε να φαίνεται
    # αν το μοντέλο μαθαίνει σταδιακά να κοιτάζει τη σωστή περιοχή.
    target_class = true_label

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))

    plot_items = []

    for ep in EPOCHS_TO_SHOW:
        ckpt_path = os.path.join(
            BEST_FOLD_CKPT_DIR,
            f"fold_{BEST_FOLD}_epoch_{ep}.pth"
        )

        if os.path.exists(ckpt_path):
            plot_items.append((f"Epoch {ep}", ckpt_path))
        else:
            print("Missing checkpoint:", ckpt_path)

    if os.path.exists(BEST_FOLD_CKPT):
        plot_items.append((f"Best epoch {best_epoch}", BEST_FOLD_CKPT))
    else:
        print("Missing best checkpoint:", BEST_FOLD_CKPT)

    if len(plot_items) == 0:
        raise FileNotFoundError("Δεν βρέθηκαν checkpoints για across epochs.")

    n_cols = len(plot_items) + 1

    plt.figure(figsize=(4.3 * n_cols, 4.8))

    plt.subplot(1, n_cols, 1)
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title(
        f"Input crop\n"
        f"True={true_class}\n"
        f"Pred={pred_class}\n"
        f"Prob={prob:.3f}"
    )

    for col_idx, (title, ckpt_path) in enumerate(plot_items, start=2):
        overlay = compute_cam_from_checkpoint(
            ckpt_path=ckpt_path,
            img_path=img_path,
            target_class=target_class
        )

        plt.subplot(1, n_cols, col_idx)
        plt.imshow(overlay)
        plt.axis("off")
        plt.title(title)

    plt.suptitle(
        f"Grad-CAM++ Across Epochs - Fold {BEST_FOLD} - {case_name}\n"
        f"Target class: {true_class}",
        fontsize=14
    )

    plt.tight_layout()

    save_path = os.path.join(
        ACROSS_EPOCHS_DIR,
        f"fold_{BEST_FOLD}_{case_name}_gradcampp_across_epochs.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)

for case_name, idx in xai_cases.items():
    plot_across_epochs(idx, case_name)

print("\nAcross-epochs Grad-CAM++ completed.")


In [ ]:
# ============================================================
# DEEPLIFT SHAP FOR BEST FOLD
# Derm7pt full + DenseNet121 + 5-fold CV
#
# Alternative to Gradient SHAP.
# Usually smoother and more readable.
# ============================================================

!pip install -q captum

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn

from captum.attr import DeepLiftShap

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

DEEPLIFT_SHAP_DIR = os.path.join(
    OUTPUT_DIR,
    "xai_outputs",
    "deeplift_shap_best_fold"
)

os.makedirs(DEEPLIFT_SHAP_DIR, exist_ok=True)

print("DeepLiftSHAP folder:", DEEPLIFT_SHAP_DIR)

# ------------------------------------------------------------
# Label map
# ------------------------------------------------------------

label_name_map = {
    0: "non_melanoma",
    1: "melanoma"
}

# ------------------------------------------------------------
# Select best fold by AUC
# ------------------------------------------------------------

best_fold_row = cv_results_df.sort_values("auc", ascending=False).iloc[0]

BEST_FOLD = int(best_fold_row["fold"])
BEST_FOLD_CKPT = best_fold_row["best_checkpoint_path"]
BEST_FOLD_PRED_PATH = best_fold_row["predictions_path"]

print("Best fold:", BEST_FOLD)
print("Best checkpoint:", BEST_FOLD_CKPT)
print("Predictions:", BEST_FOLD_PRED_PATH)

test_results_df = pd.read_csv(BEST_FOLD_PRED_PATH)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

def load_model_from_checkpoint(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    model = create_model(pretrained=False).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    return model, ckpt

best_model, best_ckpt = load_model_from_checkpoint(BEST_FOLD_CKPT)

print("Loaded best model from epoch:", best_ckpt.get("epoch", "unknown"))

# ------------------------------------------------------------
# Binary two-output wrapper
# ------------------------------------------------------------

class BinaryTwoOutputWrapper(nn.Module):
    def __init__(self, binary_model):
        super().__init__()
        self.binary_model = binary_model

    def forward(self, x):
        logit = self.binary_model(x).view(-1)

        # output 0 = non_melanoma
        # output 1 = melanoma
        return torch.stack([-logit, logit], dim=1)

wrapped_model = BinaryTwoOutputWrapper(best_model).to(device).eval()

# ------------------------------------------------------------
# Select cases
# ------------------------------------------------------------

def select_xai_cases(pred_df):
    dfp = pred_df.copy()

    dfp["pred_confidence"] = np.where(
        dfp["pred_label"] == 1,
        dfp["prob_melanoma"],
        1.0 - dfp["prob_melanoma"]
    )

    correct_melanoma = dfp[
        (dfp["true_label"] == 1) &
        (dfp["pred_label"] == 1)
    ].copy()

    correct_non_melanoma = dfp[
        (dfp["true_label"] == 0) &
        (dfp["pred_label"] == 0)
    ].copy()

    wrong = dfp[dfp["correct"] == False].copy()

    cases = {}

    if len(correct_melanoma) > 0:
        correct_melanoma["score"] = np.abs(correct_melanoma["prob_melanoma"] - 0.85)
        cases["correct_melanoma"] = correct_melanoma.sort_values("score").index[0]

    if len(correct_non_melanoma) > 0:
        correct_non_melanoma["score"] = np.abs(correct_non_melanoma["prob_melanoma"] - 0.15)
        cases["correct_non_melanoma"] = correct_non_melanoma.sort_values("score").index[0]

    if len(wrong) > 0:
        wrong["wrong_conf"] = np.where(
            wrong["pred_label"] == 1,
            wrong["prob_melanoma"],
            1.0 - wrong["prob_melanoma"]
        )

        cases["wrong_prediction"] = wrong.sort_values(
            "wrong_conf",
            ascending=False
        ).index[0]

    return cases

xai_cases = select_xai_cases(test_results_df)

print("Selected DeepLiftSHAP cases:")
print(xai_cases)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def make_deeplift_baselines(input_tensor):
    """
    DeepLiftSHAP needs multiple baselines.
    We use:
    - black baseline
    - blurred/noisy baselines
    - gray baseline
    """

    baseline_black = torch.zeros_like(input_tensor)
    baseline_gray = torch.zeros_like(input_tensor) + 0.5

    baseline_noise_1 = torch.randn_like(input_tensor) * 0.03
    baseline_noise_2 = torch.randn_like(input_tensor) * 0.06

    baselines = torch.cat(
        [
            baseline_black,
            baseline_gray,
            baseline_noise_1,
            baseline_noise_2
        ],
        dim=0
    ).to(device)

    return baselines


def compute_deeplift_shap(input_tensor, target_class):
    deeplift_shap = DeepLiftShap(wrapped_model)

    baselines = make_deeplift_baselines(input_tensor)

    attr = deeplift_shap.attribute(
        input_tensor,
        baselines=baselines,
        target=target_class
    )

    # attr: [1, 3, H, W]
    attr = attr.detach().cpu()[0]

    signed_map = attr.mean(dim=0).numpy()
    abs_map = attr.abs().mean(dim=0).numpy()

    # smoothing
    signed_map = cv2.GaussianBlur(signed_map, (7, 7), 0)
    abs_map = cv2.GaussianBlur(abs_map, (7, 7), 0)

    abs_map = (abs_map - abs_map.min()) / (
        abs_map.max() - abs_map.min() + 1e-8
    )

    return signed_map, abs_map


# ------------------------------------------------------------
# Plot DeepLiftSHAP
# ------------------------------------------------------------

def plot_deeplift_shap(idx, case_name, explain_class="pred"):
    row = test_results_df.loc[idx]

    img_path = row["image_path"]

    true_label = int(row["true_label"])
    pred_label = int(row["pred_label"])
    prob = float(row["prob_melanoma"])
    correct = bool(row["correct"])

    true_class = label_name_map[true_label]
    pred_class = label_name_map[pred_label]

    if explain_class == "pred":
        target_class = pred_label
    elif explain_class == "true":
        target_class = true_label
    else:
        target_class = int(explain_class)

    target_name = label_name_map[target_class]

    pil_img = safe_open_rgb(img_path).resize((IMG_SIZE, IMG_SIZE))
    img_np = np.array(pil_img).astype(np.float32) / 255.0

    input_tensor = eval_tf(pil_img).unsqueeze(0).to(device)

    signed_map, abs_map = compute_deeplift_shap(
        input_tensor=input_tensor,
        target_class=target_class
    )

    signed_absmax = np.max(np.abs(signed_map)) + 1e-12

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(img_np)
    plt.axis("off")
    plt.title(
        f"Input crop\n"
        f"True={true_class}\n"
        f"Pred={pred_class}"
    )

    plt.subplot(1, 4, 2)
    plt.imshow(
        signed_map,
        cmap="bwr",
        vmin=-signed_absmax,
        vmax=signed_absmax
    )
    plt.axis("off")
    plt.title(
        f"DeepLiftSHAP signed\n"
        f"Explaining={target_name}"
    )

    plt.subplot(1, 4, 3)
    plt.imshow(abs_map, cmap="jet")
    plt.axis("off")
    plt.title("Absolute importance")

    plt.subplot(1, 4, 4)
    plt.imshow(img_np)
    plt.imshow(abs_map, cmap="jet", alpha=0.35)
    plt.axis("off")
    plt.title(
        f"Overlay\n"
        f"Correct={correct}\n"
        f"Prob melanoma={prob:.3f}"
    )

    plt.tight_layout()

    save_path = os.path.join(
        DEEPLIFT_SHAP_DIR,
        f"fold_{BEST_FOLD}_{case_name}_deeplift_shap.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)


for case_name, idx in xai_cases.items():
    plot_deeplift_shap(
        idx=idx,
        case_name=case_name,
        explain_class="pred"
    )

print("\nDeepLiftSHAP completed.")